# Bangladesh Drought Monitoring - Step-by-Step Data Pipeline Test

This notebook tests the complete data extraction and processing pipeline in sequence.

## Architecture Overview

**Data Types:**
1. **STATIC** (one-time load): Admin boundaries, crop maps, soil properties
2. **HISTORICAL** (baseline for anomalies): CHIRPS 1981-2023, ERA5 historical
3. **LIVE/Near-Real-Time** (current monitoring): SMAP, MODIS, BAMIS, ENSO
4. **PROCESSING**: Compute anomalies (current vs baseline)

**Test Sequence:**
- Phase 1: Setup (Steps 1-3)
- Phase 2: Static Data (Step 4)
- Phase 3: Historical Baseline (Step 5) - OPTIONAL for quick test
- Phase 4: Live Monitoring (Steps 6-9)
- Phase 5: Processing (Steps 10-11)


In [ ]:
# STEP 1: Install Dependencies
print("="*80)
print("STEP 1/11 - Installing dependencies")
print("="*80)

import sys
import subprocess

REQUIRED_PACKAGES = [
    "earthengine-api",
    "geemap",
    "geopandas",
    "shapely",
    "pyproj",
    "fiona",
    "pyarrow",
    "lxml",
    "beautifulsoup4",
    "requests",
    "requests-cache",
    "pdfplumber",
    "matplotlib",
    "pandas",
    "numpy",
]

try:
    print("📦 Installing/validating required packages...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *REQUIRED_PACKAGES])
    print("✅ Step 1 SUCCESS: Dependencies installed")
except Exception as e:
    print(f"❌ Step 1 FAILED: {e}")
    print("➡️ Try re-running this cell. In Colab, occasional pip network issues are common.")

In [ ]:
# STEP 2: Setup GEE with Correct Project
print("="*80)
print("STEP 2/11 - Setup Google Earth Engine project")
print("="*80)

import os
import sys
from pathlib import Path

# Make sure repository root is importable in Colab.
if '/content' not in sys.path:
    sys.path.append('/content')

# If production_pipeline is in /content, this import should work directly.
# Otherwise, mount or copy your repo to /content first.
try:
    from production_pipeline.setup_colab import setup_gee_for_colab
except Exception as import_err:
    print(f"❌ Could not import setup_gee_for_colab: {import_err}")
    raise

# IMPORTANT: set your real GEE project name here.
# You can also set os.environ['GEE_PROJECT'] before running this cell.
PROJECT_NAME = os.environ.get("GEE_PROJECT", "genai-bangladesh-drought-demo")

try:
    ok = setup_gee_for_colab(PROJECT_NAME)
    print(f"✅ Step 2 SUCCESS: GEE configured with project = {PROJECT_NAME}")
    print(f"🔍 VERIFY: os.environ['GEE_PROJECT'] = {os.environ.get('GEE_PROJECT')}")
except Exception as e:
    print(f"❌ Step 2 FAILED: {e}")
    print("➡️ Verify project ID is correct and Earth Engine API is enabled for the project.")
    raise

In [ ]:
# STEP 3: Import Production Pipeline
print("="*80)
print("STEP 3/11 - Import pipeline modules")
print("="*80)

import os
import traceback
from pathlib import Path
import datetime as dt

import numpy as np
import pandas as pd

try:
    import geopandas as gpd
    from shapely.geometry import Polygon
    from shapely import wkt
except Exception:
    gpd = None
    Polygon = None
    wkt = None

try:
    from production_pipeline.extractors.static_extractor import (
        load_hdx_boundaries,
        load_mapspam_crops,
        load_soilgrids_properties,
    )
    from production_pipeline.extractors.gee_extractor import (
        extract_chirps_daily,
        extract_smap_soil_moisture,
        extract_modis_ndvi,
        extract_historical_climatology,
    )
    from production_pipeline.extractors.historical_extractor import (
        extract_chirps_historical,
        extract_era5_historical,
        compute_chirps_climatology,
        compute_era5_climatology,
        save_climatology_bundle,
        load_climatology_bundle,
        get_latest_chirps_date,
    )
    from production_pipeline.extractors.bmd_extractor import (
        scrape_bmd_spi_table,
        scrape_bmd_rainfall_7day,
    )
    from production_pipeline.extractors.bamis_extractor import (
        scrape_crop_calendars,
        scrape_bamis_bulletins,
    )
    from production_pipeline.extractors.climate_drivers import scrape_enso_indices
    from production_pipeline.processors.anomaly_calculator import compute_anomaly

    print("✅ Step 3 SUCCESS: All imports loaded")
except Exception as e:
    print(f"❌ Step 3 FAILED during imports: {e}")
    traceback.print_exc()
    raise

# Shared output folders for this notebook
BASE_DIR = Path('/content')
STATIC_DIR = BASE_DIR / 'static'
HIST_DIR = BASE_DIR / 'historical'
LIVE_DIR = BASE_DIR / 'live'
PROC_DIR = BASE_DIR / 'processed'

for p in [STATIC_DIR, HIST_DIR, LIVE_DIR, PROC_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("📁 Output folders ready:")
for p in [STATIC_DIR, HIST_DIR, LIVE_DIR, PROC_DIR]:
    print(f"   - {p}")

# Utility: robust district GeoDataFrame builder for GEE extractors

def build_districts_gdf(hdx_df: pd.DataFrame):
    """Builds a district GeoDataFrame from HDX data if possible; else fallback polygons."""
    if gpd is None or Polygon is None:
        raise RuntimeError("geopandas/shapely not available")

    # Attempt 1: geometry_wkt column from static loader
    if hdx_df is not None and not hdx_df.empty and 'geometry_wkt' in hdx_df.columns and wkt is not None:
        tmp = hdx_df.copy()
        tmp = tmp[tmp['geometry_wkt'].notna()]
        if not tmp.empty:
            try:
                tmp['geometry'] = tmp['geometry_wkt'].apply(wkt.loads)
                if 'district_id' not in tmp.columns:
                    tmp['district_id'] = tmp.index.astype(str)
                if 'district_name' not in tmp.columns:
                    tmp['district_name'] = tmp['district_id'].astype(str)
                gdf = gpd.GeoDataFrame(tmp[['district_id', 'district_name', 'geometry']].copy(), geometry='geometry', crs='EPSG:4326')
                if len(gdf) > 0:
                    return gdf
            except Exception as ge:
                print(f"⚠️ Could not parse geometry_wkt: {ge}")

    # CRITICAL: Do NOT use synthetic fallback for production downloads
    # Synthetic districts waste hours downloading data for fake locations
    raise ValueError(
        "\n\n❌ CRITICAL: No valid district boundaries loaded!\n"
        "Cannot proceed with synthetic/fallback districts.\n\n"
        "💡 FIX:\n"
        "1. Ensure production_pipeline/data/static/hdx_boundaries.csv exists\n"
        "2. Re-upload the production_pipeline folder to Colab\n"
        "3. Re-run the static data loading step\n"
    )

In [ ]:
# STEP 4: STATIC Data — Loaded from BUNDLED CSV files (no API calls)
print('='*80)
print('STEP 4/11 - STATIC data extraction (bundled CSV files)')
print('='*80)

import traceback

try:
    # ╔══════════════════════════════════════════════════════════════╗
    # ║  All three functions read from pre-bundled CSV files in     ║
    # ║  production_pipeline/data/static/                          ║
    # ║  NO API calls, NO downloads — pure local file reads.       ║
    # ╚══════════════════════════════════════════════════════════════╝

    print('\n📂 1/3  HDX Bangladesh Admin Boundaries (bundled)...')
    df_hdx = load_hdx_boundaries()
    assert len(df_hdx) == 64, f'Expected 64 districts, got {len(df_hdx)}'
    print(f'   ✅ {len(df_hdx)} districts loaded')

    print('\n📂 2/3  MAPSPAM Crop Distribution (bundled)...')
    df_mapspam = load_mapspam_crops()
    assert len(df_mapspam) == 768, f'Expected 768 rows, got {len(df_mapspam)}'
    print(f'   ✅ {len(df_mapspam)} district×crop rows loaded')

    print('\n📂 3/3  SoilGrids Soil Properties (bundled)...')
    df_soil = load_soilgrids_properties()
    assert len(df_soil) == 64, f'Expected 64 rows, got {len(df_soil)}'
    print(f'   ✅ {len(df_soil)} district rows loaded')

    # Save copies to working directory
    df_hdx.to_csv(STATIC_DIR / 'hdx_boundaries.csv', index=False)
    df_mapspam.to_csv(STATIC_DIR / 'mapspam_crops.csv', index=False)
    df_soil.to_csv(STATIC_DIR / 'soilgrids_properties.csv', index=False)

    print('\n✅ Step 4 SUCCESS: static data loaded from bundled CSVs')
    print('   Saved copies to /content/static/ for later steps')

    # ── Verification ──
    print('\n🔍 VERIFICATION CHECKS:')
    print(f'   HDX rows       : {len(df_hdx):>5} (expected: 64)    ✅' if len(df_hdx)==64 else f'   HDX rows: {len(df_hdx)} ❌')
    print(f'   MAPSPAM rows   : {len(df_mapspam):>5} (expected: 768)  ✅' if len(df_mapspam)==768 else f'   MAPSPAM rows: {len(df_mapspam)} ❌')
    print(f'   SoilGrids rows : {len(df_soil):>5} (expected: 64)    ✅' if len(df_soil)==64 else f'   SoilGrids rows: {len(df_soil)} ❌')

    has_geometry = 'geometry_wkt' in df_hdx.columns or 'geometry' in df_hdx.columns
    print(f'   HDX has geometry: {has_geometry}  ✅' if has_geometry else '   HDX has geometry: False ⚠️')

    has_canonical = 'district_id_canonical' in df_hdx.columns
    print(f'   Canonical IDs   : {has_canonical}  ✅' if has_canonical else '   Canonical IDs: missing ⚠️')

    print('\n🔍 Sample records:')
    display(df_hdx[['district_id', 'district_name']].head(5))

except Exception as e:
    print(f'❌ Step 4 FAILED: {e}')
    traceback.print_exc()


In [ ]:
# STEP 5: HISTORICAL Baseline Data
print('='*80)
print('STEP 5/11 - HISTORICAL baseline')
print('='*80)

import traceback

# ╔══════════════════════════════════════════════════════════════════╗
# ║  HISTORICAL_MODE controls how baseline data is obtained:       ║
# ║                                                                ║
# ║  'bundled' (DEFAULT) — Load pre-computed climatology CSVs      ║
# ║      from production_pipeline/data/historical/                 ║
# ║      → Instant, no GEE calls. Requires files from the         ║
# ║        Download_Historical_Baseline.ipynb notebook.            ║
# ║                                                                ║
# ║  'skip'    — Skip this step entirely. Use if you just want    ║
# ║              to test live data extraction without baselines.   ║
# ║                                                                ║
# ║  'recent'  — Download CHIRPS+ERA5 for 2015-2023 (~30 min).   ║
# ║              Good for a quick baseline if you don't have the  ║
# ║              full 1981-2023 download yet.                      ║
# ║                                                                ║
# ║  'full'    — Download CHIRPS+ERA5 for 1981-2023 (~4 hours).  ║
# ║              Use Download_Historical_Baseline.ipynb instead    ║
# ║              for better progress tracking.                     ║
# ╚══════════════════════════════════════════════════════════════════╝

HISTORICAL_MODE = 'bundled'  # Change to: 'skip', 'recent', or 'full'

try:
    # Build districts GeoDataFrame for GEE operations (needed for modes other than skip/bundled)
    hdx_local_path = STATIC_DIR / 'hdx_boundaries.csv'
    if hdx_local_path.exists():
        hdx_local = pd.read_csv(hdx_local_path)
    else:
        hdx_local = pd.DataFrame()

    districts_gdf = build_districts_gdf(hdx_local)
    print(f'🗺️ District GeoDataFrame: {len(districts_gdf)} districts')

    HIST_DIR.mkdir(parents=True, exist_ok=True)

    # ── MODE: BUNDLED ──
    if HISTORICAL_MODE == 'bundled':
        print(f'\n📂 Mode: BUNDLED — loading pre-computed climatology')
        print('   Looking in production_pipeline/data/historical/ ...')

        chirps_clim = load_climatology_bundle('chirps')
        era5_clim   = load_climatology_bundle('era5')

        if chirps_clim is not None:
            chirps_clim.to_csv(HIST_DIR / 'chirps_climatology.csv', index=False)
            print(f'   ✅ CHIRPS climatology: {len(chirps_clim):,} rows loaded')
        else:
            print('   ⚠️ No bundled CHIRPS climatology found.')
            print('      Run Download_Historical_Baseline.ipynb first, or use mode="recent".')

        if era5_clim is not None:
            era5_clim.to_csv(HIST_DIR / 'era5_climatology.csv', index=False)
            print(f'   ✅ ERA5 climatology  : {len(era5_clim):,} rows loaded')
        else:
            print('   ⚠️ No bundled ERA5 climatology found.')
            print('      Run Download_Historical_Baseline.ipynb first, or use mode="recent".')

    # ── MODE: SKIP ──
    elif HISTORICAL_MODE == 'skip':
        print('\n⏭️ Mode: SKIP — skipping historical baseline')
        existing = list(HIST_DIR.glob('*.csv'))
        if existing:
            print('   Found existing files:')
            for f in existing:
                print(f'   - {f.name}')
        else:
            print('   ⚠️ No historical files present.')
            print('   Anomaly computation (Step 10) will use proxy baselines.')

    # ── MODE: RECENT or FULL ──
    elif HISTORICAL_MODE in ('full', 'recent'):
        start_year = 1981 if HISTORICAL_MODE == 'full' else 2015
        end_year   = 2023
        est_time   = '3-4 hours' if HISTORICAL_MODE == 'full' else '~30 minutes'
        print(f'\n📥 Mode: {HISTORICAL_MODE.upper()} — downloading from GEE')
        print(f'   Period: {start_year}–{end_year}')
        print(f'   Estimated time: {est_time}')
        print(f'   💡 Tip: Use Download_Historical_Baseline.ipynb for better progress tracking.')

        # Detect latest CHIRPS date
        try:
            latest = get_latest_chirps_date()
            print(f'   Latest CHIRPS date in GEE: {latest}')
        except Exception as e:
            print(f'   ⚠️ Could not detect latest CHIRPS date: {e}')

        # ── CHIRPS ──
        print(f'\n{"─"*60}')
        print(f'CHIRPS Daily Rainfall ({start_year}–{end_year})')
        print(f'{"─"*60}')
        chirps_raw = extract_chirps_historical(
            districts_gdf,
            start_date=f'{start_year}-01-01',
            end_date=f'{end_year}-12-31',
            resume=True,
        )
        print(f'📊 CHIRPS raw data: {len(chirps_raw):,} rows')

        chirps_clim = compute_chirps_climatology(chirps_raw)
        chirps_clim.to_csv(HIST_DIR / f'chirps_climatology_{start_year}_{end_year}.csv', index=False)
        print(f'✅ CHIRPS climatology saved: {len(chirps_clim):,} rows')

        # ── ERA5 ──
        print(f'\n{"─"*60}')
        print(f'ERA5-Land Temperature ({start_year}–{end_year})')
        print(f'{"─"*60}')
        era5_raw = extract_era5_historical(
            districts_gdf,
            start_date=f'{start_year}-01-01',
            end_date=f'{end_year}-12-31',
            variables=['temperature_2m'],
            resume=True,
        )
        print(f'📊 ERA5 raw data: {len(era5_raw):,} rows')

        era5_clim = compute_era5_climatology(era5_raw)
        era5_clim.to_csv(HIST_DIR / f'era5_climatology_{start_year}_{end_year}.csv', index=False)
        print(f'✅ ERA5 climatology saved: {len(era5_clim):,} rows')

        # Save bundled copy
        save_climatology_bundle(chirps_clim, era5_clim)
        print('\n💾 Climatology saved to bundled data directory for future reuse')

    else:
        raise ValueError(f'Invalid HISTORICAL_MODE: {HISTORICAL_MODE}. Use: bundled, skip, recent, full')

    # ── Verification ──
    print(f'\n🔍 VERIFICATION — /content/historical/ inventory:')
    hist_files = sorted(HIST_DIR.glob('*'))
    if not hist_files:
        print('   (no files yet — this is OK if mode=skip)')
    for f in hist_files:
        if f.is_file():
            print(f'   - {f.name} ({f.stat().st_size/1024:.1f} KB)')

    print(f'\n✅ Step 5 complete (mode={HISTORICAL_MODE})')

except Exception as e:
    print(f'❌ Step 5 FAILED: {e}')
    traceback.print_exc()


In [ ]:
# STEP 6: LIVE - Google Earth Engine Data
print("="*80)
print("STEP 6/11 - LIVE GEE extraction (SMAP + MODIS)")
print("="*80)

import traceback
import ee

try:
    # Load districts for extraction
    hdx_local_path = STATIC_DIR / 'hdx_boundaries.csv'
    hdx_local = pd.read_csv(hdx_local_path) if hdx_local_path.exists() else pd.DataFrame()
    districts_gdf = build_districts_gdf(hdx_local)

    ee.Initialize(project=os.environ.get('GEE_PROJECT'))

    def latest_collection_date(dataset_id, band_name=None):
        coll = ee.ImageCollection(dataset_id)
        if band_name:
            coll = coll.select([band_name])
        latest = coll.sort('system:time_start', False).first()
        ts_ms = latest.get('system:time_start').getInfo()
        return pd.to_datetime(ts_ms, unit='ms').date()

    # Probe latest available dates to avoid missing-data windows and known lag issues.
    latest_smap = latest_collection_date('NASA/SMAP/SPL4SMGP/008', 'sm_surface')
    latest_modis = latest_collection_date('MODIS/061/MOD13A1', 'NDVI')

    print(f"📅 Latest SMAP date available : {latest_smap}")
    print(f"📅 Latest MODIS date available: {latest_modis}")

    smap_start = (pd.Timestamp(latest_smap) - pd.Timedelta(days=7)).strftime('%Y-%m-%d')
    smap_end = (pd.Timestamp(latest_smap) + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

    modis_start = (pd.Timestamp(latest_modis) - pd.Timedelta(days=16)).strftime('%Y-%m-%d')
    modis_end = (pd.Timestamp(latest_modis) + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

    print(f"🚀 Extracting SMAP window : {smap_start} -> {smap_end}")
    df_smap = extract_smap_soil_moisture(smap_start, smap_end, districts_gdf)

    print(f"🚀 Extracting MODIS window: {modis_start} -> {modis_end}")
    df_modis = extract_modis_ndvi(modis_start, modis_end, districts_gdf)

    smap_out = LIVE_DIR / 'smap_soil_moisture_latest.csv'
    modis_out = LIVE_DIR / 'modis_ndvi_latest.csv'

    df_smap.to_csv(smap_out, index=False)
    df_modis.to_csv(modis_out, index=False)

    print("✅ Step 6 SUCCESS: live GEE datasets saved to /content/live/")

    # VERIFY
    print("
🔍 VERIFY - data freshness")
    print(f"   SMAP max date : {pd.to_datetime(df_smap['date']).max().date() if len(df_smap) else 'N/A'}")
    print(f"   MODIS max date: {pd.to_datetime(df_modis['date']).max().date() if len(df_modis) else 'N/A'}")

    print("
🔍 VERIFY - district coverage")
    print(f"   SMAP districts : {df_smap['district_id'].nunique() if 'district_id' in df_smap.columns else 0}")
    print(f"   MODIS districts: {df_modis['district_id'].nunique() if 'district_id' in df_modis.columns else 0}")

    print("
🔍 VERIFY - sample data")
    display(df_smap.head(5))
    display(df_modis.head(5))

except Exception as e:
    print(f"❌ Step 6 FAILED: {e}")
    traceback.print_exc()

In [ ]:
# STEP 7: LIVE - Web Scraping (BMD/BAMIS)
print("="*80)
print("STEP 7/11 - LIVE scraping from BMD + BAMIS")
print("="*80)

import traceback

try:
    print("🌐 Scraping BMD SPI...")
    df_spi = scrape_bmd_spi_table()
    spi_out = LIVE_DIR / 'bmd_spi_drought.csv'
    df_spi.to_csv(spi_out, index=False)

    print("🌐 Scraping BMD 7-day rainfall...")
    df_rain = scrape_bmd_rainfall_7day()
    rain_out = LIVE_DIR / 'bmd_rainfall_7day.csv'
    df_rain.to_csv(rain_out, index=False)

    print("🌐 Scraping BAMIS crop calendars...")
    df_calendar = scrape_crop_calendars()
    cal_out = LIVE_DIR / 'bamis_crop_calendar.csv'
    df_calendar.to_csv(cal_out, index=False)

    print("🌐 Scraping BAMIS bulletins...")
    df_bulletins = scrape_bamis_bulletins()
    bul_out = LIVE_DIR / 'bamis_bulletins.csv'
    df_bulletins.to_csv(bul_out, index=False)

    print("✅ Step 7 SUCCESS: scraping outputs saved to /content/live/")

    # VERIFY
    print("
🔍 VERIFY - retrieval summary")
    print(f"   BMD SPI rows         : {len(df_spi):,}")
    print(f"   BMD rainfall rows    : {len(df_rain):,}")
    print(f"   BAMIS calendar rows  : {len(df_calendar):,}")
    print(f"   BAMIS bulletin rows  : {len(df_bulletins):,}")

    print("
🔍 VERIFY - sample records")
    display(df_spi.head(5))
    display(df_rain.head(5))
    display(df_calendar.head(5))
    display(df_bulletins.head(5))

except Exception as e:
    print(f"❌ Step 7 FAILED: {e}")
    traceback.print_exc()

In [ ]:
# STEP 8: LIVE - Climate Drivers (ENSO)
print("="*80)
print("STEP 8/11 - ENSO climate driver extraction")
print("="*80)

import traceback

try:
    df_enso = scrape_enso_indices()
    enso_out = LIVE_DIR / 'enso_indices_latest.csv'
    df_enso.to_csv(enso_out, index=False)

    print(f"✅ Step 8 SUCCESS: ENSO data saved -> {enso_out}")

    # VERIFY
    latest_row = df_enso.sort_values('date').tail(1)
    phase = latest_row['oni_phase'].iloc[0] if len(latest_row) else 'Unknown'
    oni_val = latest_row['oni_value'].iloc[0] if len(latest_row) else np.nan

    print("
🔍 VERIFY - ENSO status")
    print(f"   Current ONI phase: {phase}")
    print(f"   Current ONI value: {oni_val}")
    print(f"   Months retrieved : {len(df_enso)}")

    display(df_enso.tail(12))

except Exception as e:
    print(f"❌ Step 8 FAILED: {e}")
    traceback.print_exc()

In [ ]:
# STEP 9: DATA INVENTORY
print("="*80)
print("STEP 9/11 - Data inventory and gap check")
print("="*80)

import traceback

try:
    records = []

    for folder in [STATIC_DIR, HIST_DIR, LIVE_DIR, PROC_DIR]:
        for f in sorted(folder.glob('*')):
            if f.is_file():
                row_count = None
                try:
                    if f.suffix.lower() == '.csv':
                        row_count = len(pd.read_csv(f))
                    elif f.suffix.lower() == '.parquet':
                        row_count = len(pd.read_parquet(f))
                except Exception:
                    row_count = None

                records.append({
                    'folder': folder.name,
                    'file_name': f.name,
                    'size_kb': round(f.stat().st_size / 1024, 2),
                    'row_count': row_count,
                })

    inv_df = pd.DataFrame(records)

    if inv_df.empty:
        print("⚠️ No output files found yet.")
    else:
        print("✅ Inventory generated")
        display(inv_df.sort_values(['folder', 'file_name']).reset_index(drop=True))

    expected_live = {
        'smap_soil_moisture_latest.csv',
        'modis_ndvi_latest.csv',
        'bmd_spi_drought.csv',
        'bmd_rainfall_7day.csv',
        'bamis_crop_calendar.csv',
        'bamis_bulletins.csv',
        'enso_indices_latest.csv',
    }
    found_live = {f.name for f in LIVE_DIR.glob('*.csv')}
    missing_live = sorted(expected_live - found_live)

    print("
🔍 VERIFY - gap check")
    if missing_live:
        print("⚠️ Missing expected LIVE files:")
        for m in missing_live:
            print(f"   - {m}")
    else:
        print("✅ All expected LIVE files are present")

except Exception as e:
    print(f"❌ Step 9 FAILED: {e}")
    traceback.print_exc()

In [ ]:
# STEP 10: PROCESSING - Anomaly Computation
print("="*80)
print("STEP 10/11 - Compute anomalies (current vs baseline)")
print("="*80)

import traceback

try:
    # -----------------------------
    # Load current live conditions
    # -----------------------------
    smap_file = LIVE_DIR / 'smap_soil_moisture_latest.csv'
    modis_file = LIVE_DIR / 'modis_ndvi_latest.csv'

    if not smap_file.exists() or not modis_file.exists():
        raise FileNotFoundError("Required live files missing. Run Step 6 first.")

    smap_live = pd.read_csv(smap_file)
    modis_live = pd.read_csv(modis_file)

    smap_live['date'] = pd.to_datetime(smap_live['date'], errors='coerce')
    modis_live['date'] = pd.to_datetime(modis_live['date'], errors='coerce')

    smap_current = (
        smap_live.sort_values('date')
        .groupby(['district_id', 'district_name'], as_index=False)
        .tail(1)[['district_id', 'district_name', 'sm_surface']]
        .rename(columns={'sm_surface': 'soil_moisture_m3_m3'})
    )

    modis_current = (
        modis_live.sort_values('date')
        .groupby(['district_id', 'district_name'], as_index=False)
        .tail(1)[['district_id', 'district_name', 'ndvi']]
    )

    # --------------------------------------
    # Load/derive baselines for anomalies
    # --------------------------------------
    def load_first_existing(paths):
        for p in paths:
            if Path(p).exists():
                return pd.read_csv(p), Path(p)
        return None, None

    smap_base_df, smap_base_path = load_first_existing([
        HIST_DIR / 'smap_climatology.csv',
        HIST_DIR / 'smap_climatology_2015_2023.csv',
    ])

    ndvi_base_df, ndvi_base_path = load_first_existing([
        HIST_DIR / 'ndvi_baseline.csv',
        HIST_DIR / 'modis_ndvi_baseline.csv',
        HIST_DIR / 'modis_climatology.csv',
    ])

    # Soil baseline handling
    if smap_base_df is not None and {'district_id', 'var_mean', 'var_std'}.issubset(smap_base_df.columns):
        smap_baseline = (
            smap_base_df.groupby('district_id', as_index=False)[['var_mean', 'var_std']]
            .mean()
            .rename(columns={'var_mean': 'baseline_mean', 'var_std': 'baseline_std'})
        )
        print(f"✅ Soil baseline loaded from: {smap_base_path}")
    else:
        print("⚠️ No SMAP historical baseline found. Building proxy baseline from current snapshot.")
        mu = smap_current['soil_moisture_m3_m3'].mean()
        sd = smap_current['soil_moisture_m3_m3'].std()
        sd = float(sd) if pd.notna(sd) and sd > 1e-6 else 1.0
        smap_baseline = smap_current[['district_id']].copy()
        smap_baseline['baseline_mean'] = mu
        smap_baseline['baseline_std'] = sd

    # NDVI baseline handling
    if ndvi_base_df is not None:
        if {'district_id', 'baseline_mean', 'baseline_std'}.issubset(ndvi_base_df.columns):
            ndvi_baseline = ndvi_base_df[['district_id', 'baseline_mean', 'baseline_std']].copy()
            print(f"✅ NDVI baseline loaded from: {ndvi_base_path}")
        elif {'district_id', 'var_mean', 'var_std'}.issubset(ndvi_base_df.columns):
            ndvi_baseline = (
                ndvi_base_df.groupby('district_id', as_index=False)[['var_mean', 'var_std']]
                .mean()
                .rename(columns={'var_mean': 'baseline_mean', 'var_std': 'baseline_std'})
            )
            print(f"✅ NDVI climatology converted from: {ndvi_base_path}")
        else:
            ndvi_base_df = None

    if ndvi_base_df is None:
        print("⚠️ No NDVI historical baseline found. Building proxy baseline from current snapshot.")
        mu = modis_current['ndvi'].mean()
        sd = modis_current['ndvi'].std()
        sd = float(sd) if pd.notna(sd) and sd > 1e-6 else 0.05
        ndvi_baseline = modis_current[['district_id']].copy()
        ndvi_baseline['baseline_mean'] = mu
        ndvi_baseline['baseline_std'] = sd

    # -----------------------------
    # Compute anomalies
    # -----------------------------
    soil_anom = compute_anomaly(
        current_df=smap_current,
        baseline_df=smap_baseline,
        key_cols=['district_id'],
        metric_col='soil_moisture_m3_m3',
        output_prefix='soil_moisture',
    )

    ndvi_anom = compute_anomaly(
        current_df=modis_current,
        baseline_df=ndvi_baseline,
        key_cols=['district_id'],
        metric_col='ndvi',
        output_prefix='ndvi',
    )

    def classify_from_z(z):
        if pd.isna(z):
            return 'Unknown'
        if z <= -2.0:
            return 'Severe Negative Anomaly'
        if z <= -1.0:
            return 'Moderate Negative Anomaly'
        if z < 1.0:
            return 'Near Normal'
        if z < 2.0:
            return 'Moderate Positive Anomaly'
        return 'Strong Positive Anomaly'

    soil_anom['anomaly_class'] = soil_anom['soil_moisture_anomaly_z'].apply(classify_from_z)
    ndvi_anom['anomaly_class'] = ndvi_anom['ndvi_anomaly_z'].apply(classify_from_z)

    # Save outputs
    soil_out = PROC_DIR / 'soil_moisture_anomaly.csv'
    ndvi_out = PROC_DIR / 'ndvi_anomaly.csv'

    soil_anom.to_csv(soil_out, index=False)
    ndvi_anom.to_csv(ndvi_out, index=False)

    print("✅ Step 10 SUCCESS: anomaly outputs saved to /content/processed/")

    # VERIFY
    print("
🔍 VERIFY - anomaly class distribution")
    print("Soil moisture anomaly classes:")
    display(soil_anom['anomaly_class'].value_counts(dropna=False).rename_axis('class').reset_index(name='count'))

    print("NDVI anomaly classes:")
    display(ndvi_anom['anomaly_class'].value_counts(dropna=False).rename_axis('class').reset_index(name='count'))

    print("
🔍 VERIFY - sample outputs")
    display(soil_anom.head(5))
    display(ndvi_anom.head(5))

except Exception as e:
    print(f"❌ Step 10 FAILED: {e}")
    traceback.print_exc()

In [ ]:
# STEP 11: FINAL VALIDATION
print("="*80)
print("STEP 11/11 - Final validation report")
print("="*80)

import traceback

try:
    expected_files = {
        'static': [
            STATIC_DIR / 'hdx_boundaries.csv',
            STATIC_DIR / 'mapspam_crops.csv',
            STATIC_DIR / 'soilgrids_properties.csv',
        ],
        'live': [
            LIVE_DIR / 'smap_soil_moisture_latest.csv',
            LIVE_DIR / 'modis_ndvi_latest.csv',
            LIVE_DIR / 'bmd_spi_drought.csv',
            LIVE_DIR / 'bmd_rainfall_7day.csv',
            LIVE_DIR / 'bamis_crop_calendar.csv',
            LIVE_DIR / 'bamis_bulletins.csv',
            LIVE_DIR / 'enso_indices_latest.csv',
        ],
        'processed': [
            PROC_DIR / 'soil_moisture_anomaly.csv',
            PROC_DIR / 'ndvi_anomaly.csv',
        ],
    }

    rows = []
    issues = []

    for phase, files in expected_files.items():
        for f in files:
            exists = f.exists()
            nrows = None
            if exists:
                try:
                    if f.suffix == '.csv':
                        nrows = len(pd.read_csv(f))
                except Exception as re:
                    issues.append(f"Read issue: {f.name} -> {re}")

            rows.append({
                'phase': phase,
                'file': str(f),
                'exists': exists,
                'rows': nrows,
            })

            if not exists:
                issues.append(f"Missing file: {f}")
            elif nrows is not None and nrows == 0:
                issues.append(f"Empty file: {f}")

    summary_df = pd.DataFrame(rows)

    print("📊 DATA AVAILABILITY MATRIX")
    display(summary_df)

    print("
🧪 PIPELINE HEALTH SUMMARY")
    total_expected = len(summary_df)
    total_present = int(summary_df['exists'].sum())
    total_non_empty = int(((summary_df['exists']) & (summary_df['rows'].fillna(1) > 0)).sum())

    print(f"   Expected files : {total_expected}")
    print(f"   Present files  : {total_present}")
    print(f"   Non-empty files: {total_non_empty}")

    if issues:
        print("
⚠️ Issues flagged:")
        for i, issue in enumerate(issues, 1):
            print(f"   {i}. {issue}")
        print("
➡️ Recommended next steps:")
        print("   - Re-run failed step cells shown above")
        print("   - For GEE issues: verify project + authentication")
        print("   - For scraping issues: website structure may have changed; inspect HTML snapshots")
        print("   - For anomaly issues: ensure historical baseline files are available")
    else:
        print("
✅ FINAL RESULT: End-to-end pipeline test completed successfully")
        print("➡️ Recommended next step: convert this tested flow into scheduled production scripts")

except Exception as e:
    print(f"❌ Step 11 FAILED: {e}")
    traceback.print_exc()